In [ ]:
#| default_exp scheduler

# Reviewing

> The v3 scheduler: card states, answer buttons, and the revlog, compatible with every Anki client

This module is a port of the scheduling half of Anki's Rust `scheduler` module:
- the state machine that moves cards between new, learning, review and relearning
- the SM-2 interval arithmetic
- the bookkeeping an answer leaves behind (the card row, a `revlog` entry, deck daily counters, leech handling)

The review log is the durable source of truth in Anki's design: any client can rebuild scheduling state from it, so the revlog rows we write are what make a fastanki review indistinguishable from one done on e.g. the desktop or mobile app.

Like the rest of fastanki, every claim here is checked against Anki itself: the `anki` package answers the same cards under the same config, and we compare its stored scheduling states, card rows and revlog entries with ours. Setting `ANKI_TEST_MODE` before the oracle loads disables its interval fuzz, making those comparisons exact.

In [ ]:
import json, time, random
from datetime import datetime, timezone
from collections import namedtuple
from fastcore.utils import *
from fastanki.schema import *
from fastanki.collection import *
from fastanki._proto import deck_config_pb2, decks_pb2
from fastanki.fsrs import *

In [ ]:
import os, tempfile, shutil
from fastcore.test import *

In [ ]:
os.environ['ANKI_TEST_MODE'] = '1'   # must precede the oracle's first scheduling call: disables its interval fuzz

In [ ]:
from anki.collection import Collection as AnkiCollection
from anki.scheduler.v3 import CardAnswer

## Deck presets

Scheduling is configured per deck preset: learning steps, ease multipliers, daily limits, leech handling. The settings live in the `deck_config` table as a protobuf blob whose field names already say what they mean, so rather than wrap it, we hand back the parsed `DeckConfig.Config` message directly. A deck names its preset inside its `kind` blob.

In [ ]:
@patch
def deck_conf(self:Collection, did):
    "The parsed `DeckConfig.Config` preset governing deck `did`"
    k = decks_pb2.Deck.KindContainer()
    k.ParseFromString(self.q1('select kind from decks where id=?', did))
    blob = self.q1('select config from deck_config where id=?', k.normal.config_id or 1) or default_deck_config()
    c = deck_config_pb2.DeckConfig.Config()
    c.ParseFromString(blob)
    return c

In [ ]:
td = Path(tempfile.mkdtemp())
col = Collection.open(td/'collection.anki2')
cfg = col.deck_conf(1)
test_eq(list(cfg.learn_steps), [1.0,10.0])
test_eq((cfg.leech_threshold, cfg.maximum_review_interval), (8, 36500))
cfg.graduating_interval_good, cfg.initial_ease, cfg.hard_multiplier

(1, 2.5, 1.2000000476837158)

## Card states

A card is always in one of four states, and every answer maps a state to a new state.
- `NewSt` waits in the new queue at a `position`
- `LearnSt` is inside the (re)learning steps with `remaining` steps left and `secs` until the next showing
- `ReviewSt` has graduated and comes back every `ivl` days, its `ease` deciding how fast that grows
- `RelearnSt` is a failed review: a `LearnSt` to work through plus the `ReviewSt` to return to.

They are named tuples, so states compare by value in tests.

Intervals travel in the revlog's own convention: a non-negative number is days, a negative number is `-seconds`. `ivl_days` is Anki's `maybe_as_days`: a seconds interval that crosses the next day boundary becomes a day count, which is how a `1d` learning step lands in the day-learn queue rather than 24 hours of wall clock.

In [ ]:
class NewSt(namedtuple('NewSt', 'position')):
    "Not yet studied; `position` orders the new queue"
    def ivl_kind(self):    return 0
    def revlog_kind(self): return 0

class LearnSt(namedtuple('LearnSt', 'remaining secs elapsed mem', defaults=(0,None))):
    "In the learning steps: `remaining` steps left, `secs` until the next showing"
    def ivl_kind(self):    return -self.secs
    def revlog_kind(self): return 0

class ReviewSt(namedtuple('ReviewSt', 'ivl ease lapses elapsed leeched mem', defaults=(0,0,False,None))):
    "Graduated: due again in `ivl` days, growing by `ease`"
    def ivl_kind(self):    return self.ivl
    def revlog_kind(self): return 3 if self.elapsed<self.ivl else 1

class RelearnSt(namedtuple('RelearnSt', 'learn review')):
    "A failed review: `learn` steps to redo, then back to `review`"
    def ivl_kind(self):    return self.learn.ivl_kind()
    def revlog_kind(self): return 2

class Answers(namedtuple('Answers', 'current again hard good easy')):
    "One state per answer button (index by ease 1-4), plus the current state"

def ivl_days(iv, secs_to_rollover):
    "Convert a seconds interval that crosses the day boundary to days (Anki's `maybe_as_days`)"
    if iv<0 and -iv>=secs_to_rollover: return (-iv-secs_to_rollover)//86400 + 1
    return iv

def ivl_secs(iv):
    "An interval in plain seconds"
    return -iv if iv<0 else iv*86400

In [ ]:
test_eq(ivl_days(-600, 3600), -600)              # 10min tonight stays seconds
test_eq(ivl_days(-86400, 3600), 1)               # a 1d step becomes day-learn: due tomorrow
test_eq(ivl_days(-90000, 3600), 2)               # ...or the day after, if little of today remains
test_eq(ReviewSt(10, 2.5).revlog_kind(), 3)      # reviewed early: logged as kind 3 like Anki
test_eq(ReviewSt(10, 2.5, elapsed=10).revlog_kind(), 1)
Answers(*'cahge')[3]

'g'

## Learning steps

The learning-steps arithmetic ports `steps.rs`. Steps are minutes; a card's `remaining` count says how many are left (the stored `left` column may carry a legacy thousands part, so it's taken mod 1000). Two quirks are Anki's own: on the *first* step, Hard averages the first two steps (or takes 1.5x the only step, capped a day above it) so it lands between Again and Good; and any delay past a day is rounded to whole days so morning and evening study give the same answer.

In [ ]:
DAY = 86400

def _round(x):
    "Round half away from zero, like Rust's `round` (Python's `round` is banker's)"
    return int(x+0.5) if x>=0 else -int(-x+0.5)

def _step_secs(steps, i): return int(steps[i]*60) if 0<=i<len(steps) else None
def _step_idx(steps, remaining): return clamp(len(steps)-remaining%1000, 0, max(len(steps)-1, 0))
def _round_days(secs): return _round(secs/DAY)*DAY if secs>DAY else secs

def step_again_delay(steps): return _step_secs(steps, 0)

def step_hard_delay(steps, remaining):
    idx = _step_idx(steps, remaining)
    cur = _step_secs(steps, idx) or _step_secs(steps, 0)
    if cur is None: return None
    if idx>0: return cur
    nxt = _step_secs(steps, 1)
    if nxt is not None: return _round_days((cur+nxt)//2)
    return _round_days(min(cur*3//2, cur+DAY))

def step_good_delay(steps, remaining): return _step_secs(steps, _step_idx(steps, remaining)+1)
def step_current_delay(steps, remaining): return _step_secs(steps, _step_idx(steps, remaining)) or 0
def step_remaining_good(steps, remaining): return len(steps)-_step_idx(steps, remaining)-1

These vectors are `steps.rs`'s own test suite: a lone 10-minute step, a 3-day step (Hard capped at 4 days), and two- and three-step ladders at each position.

In [ ]:
def _delays(steps, remaining): return (step_again_delay(steps), step_hard_delay(steps, remaining), step_good_delay(steps, remaining))
test_eq(_delays([10.0], 1), (600, 900, None))
test_eq(_delays([3*DAY/60], 1), (3*DAY, 4*DAY, None))
test_eq(_delays([1.0,10.0], 2), (60, 330, 600))
test_eq(_delays([1.0,10.0], 1), (60, 600, None))
test_eq(_delays([1.0,10.0,100.0], 3), (60, 330, 600))
test_eq(_delays([1.0,10.0,100.0], 2), (60, 600, 6000))
test_eq(_delays([1.0,10.0,100.0], 1), (60, 6000, None))
test_eq(step_remaining_good([1.0,10.0], 2), 1)
test_eq(step_current_delay([1.0,10.0], 1), 600)

## Interval fuzz

Anki nudges every review interval by a small random amount so cards added together don't stay clumped forever. The fuzz *bounds* port `fuzz.rs` exactly: nothing below 2.5 days, then ±1 day plus a sliding percentage of the days in each range. The *pick* within those bounds comes from a per-`(card, reps)` seeded generator, so re-computing a card's schedule is deterministic.

Note: Rust and Python use different RNGs, so with the same seed the two implementations pick different (equally valid) points inside identical bounds; with fuzz disabled (`ANKI_TEST_MODE`), the two agree exactly. `learn_fuzz` is the separate, smaller fuzz applied to intraday learning delays: up to 25% extra, capped at 5 minutes.

In [ ]:
_FUZZ_RANGES = [(2.5,7.0,0.15), (7.0,20.0,0.1), (20.0,None,0.05)]

def fuzz_delta(ivl):
    "Days of fuzz applied either side of `ivl`"
    if ivl<2.5: return 0.0
    d = 1.0
    for start,end,f in _FUZZ_RANGES: d += f*max(min(ivl, end or ivl)-start, 0)
    return d

def fuzz_bounds(
    ivl,      # Undisturbed interval in days (may be fractional)
    lo=1,     # Minimum permitted result
    hi=36500, # Maximum permitted result
):
    "Inclusive `(lower, upper)` day bounds for a fuzzed interval, respecting `lo`/`hi`"
    lo = min(lo, hi)
    ivl = clamp(ivl, lo, hi)
    d = fuzz_delta(ivl)
    l,u = _round(ivl-d), _round(ivl+d)
    l,u = clamp(l, lo, hi), clamp(u, lo, hi)
    if u==l and u>2 and u<hi: u = l+1
    return l,u

def with_fuzz(fz, ivl, lo, hi):
    "`ivl` fuzzed by factor `fz` in [0,1) within `lo`/`hi`; None means round and clamp only"
    if fz is None: return clamp(_round(ivl), lo, hi)
    l,u = fuzz_bounds(ivl, lo, hi)
    return int(l + fz*(1+u-l))

def fuzz_factor(cid, reps):
    "Deterministic per-(card, rep) fuzz factor in [0,1)"
    return random.Random(cid+reps).random()

def min_fuzz_ivl(ivl, prev_ivl, max_ivl):
    "Minimum for a fuzzed FSRS review interval: fuzz may not shrink an interval that grew (Anki's `minimum_review_fuzz_interval`)"
    upper = fuzz_bounds(ivl, 1, max_ivl)[1]
    if _round(ivl) > prev_ivl: return prev_ivl+1
    return prev_ivl if prev_ivl <= upper else 0

def learn_fuzz(fz_seed, secs):
    "Intraday learning delay with Anki's up-to-25% (max 5min) extension; `fz_seed` None leaves it alone"
    if fz_seed is None: return secs
    upper = secs + int(min(secs*0.25, 300.0))
    if secs >= upper: return secs
    return random.Random(fz_seed).randrange(secs, upper)

`fuzz.rs`'s vectors, driven at fuzz factors 0, 0.5 and 0.99 to hit each bound and the middle:

In [ ]:
def _lmu(ivl, lo, hi): return tuple(with_fuzz(f, ivl, lo, hi) for f in (0.0, 0.5, 0.99))
test_eq(with_fuzz(None, 1.5, 1, 100), 2)
test_eq(with_fuzz(None, 101.0, 1, 100), 100)
test_eq(_lmu(1.0, 1, 1000), (1,1,1))        # no fuzz below 2.5 days
test_eq(_lmu(2.5, 1, 1000), (2,3,4))        # then 1 day either side
test_eq(_lmu(7.0, 1, 1000), (5,7,9))        # plus 0.15/day in 2.5-7
test_eq(_lmu(17.0, 1, 1000), (14,17,20))    # plus 0.1/day in 7-20
test_eq(_lmu(37.0, 1, 1000), (33,37,41))    # plus 0.05/day above 20
test_eq(_lmu(2.0, 2, 1000), (2,2,2))
test_eq(_lmu(2.0, 3, 1000), (3,4,4))        # widened to a 2-day range when bounds allow
test_eq(_lmu(2.0, 3, 3), (3,3,3))
test_eq(_lmu(19.9, 3, 1000), (17,20,23))
test_eq(learn_fuzz(None, 600), 600)
assert 600 <= learn_fuzz(42, 600) < 750
test_eq(learn_fuzz(42, 600), learn_fuzz(42, 600))   # seeded: recomputable

## The state machine

`Ctx` carries what a transition needs from the deck preset, plus the per-answer fuzz factor (`fsrs` stays None for SM-2; a later section fills it). Each state's `next_answers` returns an `Answers` of the four button outcomes, a direct port of `review.rs`, `learning.rs` and `relearning.rs`:

- A failing review multiplies its interval by `lapse_mult` (0 by default: start over), drops ease by 0.2, and enters relearning if there are relearn steps. Lapses at the leech threshold — and every half-threshold after — mark the card `leeched`.
- A passing review scales by `hard_mult`, `ease`, or `ease * easy_mult`, with overdue days credited at half weight for Good and full weight for Easy, each button forced at least a day past the previous one. Reviewed *early* (`elapsed < ivl`, only reachable through filtered decks — for normal decks `card_state` clamps due to today — but ported for completeness), elapsed days take the place of scheduled days and no fuzz applies.
- Learning cards walk the steps: Again restarts them, Hard repeats (with the first-step average quirk), Good advances, Easy graduates straight to `grad_easy` days. Past the last step, Good graduates to `grad_good`.
- New cards answer exactly like a learning card that just failed: full steps remaining.

In [ ]:
_CTX = 'fuzz steps relearn_steps grad_good grad_easy init_ease hard_mult easy_mult ivl_mult lapse_mult max_ivl min_lapse_ivl leech_threshold fsrs allow_short short_steps'
class Ctx(namedtuple('Ctx', _CTX, defaults=(None,False,False))):
    "Everything a state transition needs: the deck preset's numbers plus this answer's fuzz factor"

def mk_ctx(cfg, fuzz=None, fsrs=None, **over):
    "A `Ctx` from deck preset `cfg`, overridable by keyword"
    d = dict(fuzz=fuzz, steps=list(cfg.learn_steps), relearn_steps=list(cfg.relearn_steps),
        grad_good=cfg.graduating_interval_good, grad_easy=cfg.graduating_interval_easy, init_ease=cfg.initial_ease,
        hard_mult=cfg.hard_multiplier, easy_mult=cfg.easy_multiplier, ivl_mult=cfg.interval_multiplier,
        lapse_mult=cfg.lapse_multiplier, max_ivl=cfg.maximum_review_interval,
        min_lapse_ivl=cfg.minimum_lapse_interval, leech_threshold=cfg.leech_threshold, fsrs=fsrs)
    d.update(over)
    return Ctx(**d)

def _min_max(ctx, minimum):
    hi = max(ctx.max_ivl, 1)
    return clamp(minimum, 1, hi), hi

def leech_threshold_met(lapses, threshold):
    "True at `threshold` lapses, and every half-threshold (rounded up) after"
    if not threshold: return False
    half = max(-(-threshold//2), 1)
    return lapses>=threshold and (lapses-threshold)%half==0

In [ ]:
EASE_AGAIN,EASE_HARD,EASE_EASY,MIN_EASE = -0.2,-0.15,0.15,1.3

def _constrain_passing(ctx, ivl, minimum, fuzz=True):
    if ctx.fsrs is None: ivl *= ctx.ivl_mult
    lo,hi = _min_max(ctx, minimum)
    return with_fuzz(ctx.fuzz, ivl, lo, hi) if fuzz else clamp(_round(ivl), lo, hi)

def _mem(ctx, rating):
    "The FSRS memory state this rating leads to (None under SM-2)"
    return ctx.fsrs[rating-1].mem if ctx.fsrs is not None else None

@patch
def _passing_ivls(self:ReviewSt, ctx):
    "Hard/good/easy intervals, each at least a day past the one before"
    if ctx.fsrs is not None: return self._passing_fsrs_ivls(ctx)
    if self.elapsed < self.ivl: return self._passing_early_ivls(ctx)
    cur,late = max(self.ivl,1), max(self.elapsed-self.ivl, 0)
    hard_min = 0 if ctx.hard_mult<=1.0 else self.ivl+1
    hard = _constrain_passing(ctx, cur*ctx.hard_mult, hard_min)
    good_min = self.ivl+1 if ctx.hard_mult<=1.0 else hard+1
    good = _constrain_passing(ctx, (cur+late/2)*self.ease, good_min)
    easy = _constrain_passing(ctx, (cur+late)*self.ease*ctx.easy_mult, good+1)
    return hard,good,easy

@patch
def _passing_fsrs_ivls(self:ReviewSt, ctx):
    "FSRS intervals come straight from the memory model; fuzz may not shrink an interval that grew"
    ivls = [s.ivl for s in ctx.fsrs]
    hard = _constrain_passing(ctx, ivls[1], max(min_fuzz_ivl(ivls[1], self.ivl, ctx.max_ivl), 1))
    good = _constrain_passing(ctx, ivls[2], max(min_fuzz_ivl(ivls[2], self.ivl, ctx.max_ivl), hard+1))
    easy = _constrain_passing(ctx, ivls[3], max(min_fuzz_ivl(ivls[3], self.ivl, ctx.max_ivl), good+1))
    return hard,good,easy

@patch
def _passing_early_ivls(self:ReviewSt, ctx):
    "Reviewed before due: elapsed days stand in for scheduled, no fuzz"
    sched,elap = max(self.ivl,1), self.elapsed
    hard = _constrain_passing(ctx, max(elap*ctx.hard_mult, sched*ctx.hard_mult/2), 0, fuzz=False)
    good = _constrain_passing(ctx, max(elap*self.ease, sched), 0, fuzz=False)
    bonus = ctx.easy_mult - (ctx.easy_mult-1.0)/2
    easy = _constrain_passing(ctx, max(elap*self.ease, sched)*bonus, 0, fuzz=False)
    return hard,good,easy

@patch
def _failing_ivl(self:ReviewSt, ctx):
    if ctx.fsrs is not None: return ctx.fsrs[0].ivl   # in FSRS, fuzz applies when leaving relearning
    lo,hi = _min_max(ctx, ctx.min_lapse_ivl)
    return with_fuzz(ctx.fuzz, max(self.ivl,1)*ctx.lapse_mult, lo, hi)

@patch
def next_answers(self:ReviewSt, ctx):
    hard,good,easy = self._passing_ivls(ctx)
    lapses = self.lapses+1
    fail = self._failing_ivl(ctx)
    days = max(_round(max(fail,0)), 1)
    again_review = ReviewSt(days, max(self.ease+EASE_AGAIN, MIN_EASE), lapses, mem=_mem(ctx,1),
        leeched=leech_threshold_met(lapses, ctx.leech_threshold))
    if ctx.relearn_steps:
        again = RelearnSt(LearnSt(len(ctx.relearn_steps), step_again_delay(ctx.relearn_steps), mem=_mem(ctx,1)), again_review)
    elif ctx.fsrs is not None and ctx.allow_short and (ctx.short_steps or not ctx.relearn_steps) and fail < 0.5:
        again = RelearnSt(LearnSt(0, int(fail*86400), mem=_mem(ctx,1)), again_review)
    else: again = again_review
    return Answers(self, again, ReviewSt(hard, max(self.ease+EASE_HARD, MIN_EASE), self.lapses, mem=_mem(ctx,2)),
        ReviewSt(good, self.ease, self.lapses, mem=_mem(ctx,3)),
        ReviewSt(easy, self.ease+EASE_EASY, self.lapses, mem=_mem(ctx,4)))

In [ ]:
def _graduate(ctx, rating):
    "Leave the learning steps for review: `grad_good`/`grad_easy` days under SM-2, the model's interval under FSRS"
    lo,hi = _min_max(ctx, 1)
    if ctx.fsrs is None:
        ivl = ctx.grad_easy if rating==4 else ctx.grad_good
        return ReviewSt(with_fuzz(ctx.fuzz, max(_round(ivl),1), lo, hi), ctx.init_ease)
    st = ctx.fsrs[rating-1]
    if rating==4: lo = with_fuzz(ctx.fuzz, ctx.fsrs[2].ivl, lo, hi) + 1   # Easy must clear the fuzzed Good interval
    return ReviewSt(with_fuzz(ctx.fuzz, max(_round(st.ivl),1), lo, hi), ctx.init_ease, mem=st.mem)

def _learn_short(ctx, rating, steps):
    "The FSRS short-term state when the model wants this answer back the same day, else None"
    if ctx.fsrs is None or not ctx.allow_short or not (ctx.short_steps or not steps): return None
    st = ctx.fsrs[rating-1]
    return st if st.ivl < 0.5 else None

@patch
def next_answers(self:LearnSt, ctx):
    steps = ctx.steps
    def grad_or_short(rating, remaining):
        s = _learn_short(ctx, rating, steps)
        if s is not None: return LearnSt(remaining, int(s.ivl*86400), mem=s.mem)
        return _graduate(ctx, rating)
    ad = step_again_delay(steps)
    again = LearnSt(len(steps), ad, mem=_mem(ctx,1)) if ad is not None else grad_or_short(1, len(steps))
    hd = step_hard_delay(steps, self.remaining)
    hard = LearnSt(self.remaining, hd, mem=_mem(ctx,2)) if hd is not None else grad_or_short(2, self.remaining)
    gd = step_good_delay(steps, self.remaining)
    good = LearnSt(step_remaining_good(steps, self.remaining), gd, mem=_mem(ctx,3)) if gd is not None else grad_or_short(3, self.remaining)
    return Answers(self, again, hard, good, _graduate(ctx, 4))

@patch
def next_answers(self:NewSt, ctx):
    "A new card answers like a learning card that just failed"
    return LearnSt(len(ctx.steps), 0).next_answers(ctx)._replace(current=self)

def _relearn_pass(rl, ctx, rating):
    "A passing FSRS answer in relearning: back to review, or another same-day step if the model wants one"
    lo,hi = _min_max(ctx, 1)
    st = ctx.fsrs[rating-1]
    rev = rl.review._replace(ivl=with_fuzz(ctx.fuzz, max(_round(st.ivl),1), lo, hi), mem=st.mem)
    if ctx.allow_short and (ctx.short_steps or not ctx.relearn_steps) and st.ivl < 0.5:
        rem = rl.learn.remaining if rating==2 else step_remaining_good(ctx.relearn_steps, rl.learn.remaining)
        return RelearnSt(rl.learn._replace(remaining=rem, secs=int(st.ivl*86400), elapsed=0, mem=st.mem), rev)
    return rev

@patch
def next_answers(self:RelearnSt, ctx):
    steps,rev,fs = ctx.relearn_steps, self.review, ctx.fsrs
    fail = rev._failing_ivl(ctx)
    days = max(_round(max(fail,0)), 1)
    ad = step_again_delay(steps)
    if ad is not None: again = RelearnSt(LearnSt(len(steps), ad, mem=_mem(ctx,1)), rev._replace(ivl=days, elapsed=0, mem=_mem(ctx,1)))
    elif fs is not None:
        lo,hi = _min_max(ctx, 1)
        again_rev = rev._replace(ivl=with_fuzz(ctx.fuzz, max(_round(fail),1), lo, hi), mem=fs[0].mem)
        if ctx.allow_short and (ctx.short_steps or not steps) and fail < 0.5:
            again = RelearnSt(LearnSt(len(steps), int(fail*86400), mem=fs[0].mem), again_rev)
        else: again = again_rev
    else: again = rev
    hd = step_hard_delay(steps, self.learn.remaining)
    if hd is not None: hard = RelearnSt(self.learn._replace(secs=hd, elapsed=0, mem=_mem(ctx,2)), rev._replace(elapsed=0, mem=_mem(ctx,2)))
    elif fs is not None: hard = _relearn_pass(self, ctx, 2)
    else: hard = rev
    gd = step_good_delay(steps, self.learn.remaining)
    if gd is not None:
        good = RelearnSt(LearnSt(step_remaining_good(steps, self.learn.remaining), gd, mem=_mem(ctx,3)), rev._replace(elapsed=0, mem=_mem(ctx,3)))
    elif fs is not None: good = _relearn_pass(self, ctx, 3)
    else: good = rev
    if fs is not None:
        lo,hi = _min_max(ctx, 1)
        lo = with_fuzz(ctx.fuzz, fs[2].ivl, lo, hi) + 1
        easy = rev._replace(ivl=with_fuzz(ctx.fuzz, max(_round(fs[3].ivl),1), lo, hi), elapsed=0, mem=fs[3].mem)
    else: easy = rev._replace(ivl=rev.ivl+1, elapsed=0)
    return Answers(self, again, hard, good, easy)

The checks below are `review.rs`'s own unit tests: leech cadence at whole and half thresholds, the low-ease/low-multiplier interval ladder at fuzz 0 and 0.99, a silly 0.1x multiplier that must not underflow, and the maximum interval clamping everything to 5 days.

In [ ]:
for lapses,want in [(2,False),(3,True),(4,False),(5,True),(6,False),(7,True)]: test_eq(leech_threshold_met(lapses,3), want)
for lapses,want in [(7,False),(8,True),(9,False),(11,False),(12,True)]: test_eq(leech_threshold_met(lapses,8), want)
test_eq(leech_threshold_met(0,0), False)
for lapses,want in [(0,False),(1,True),(2,True),(3,True)]: test_eq(leech_threshold_met(lapses,1), want)

_ctx = mk_ctx(cfg, fuzz=0.0)
st = ReviewSt(1, 1.3, elapsed=1)
test_eq(st._passing_ivls(_ctx), (2,3,4))
test_eq(st._passing_ivls(_ctx._replace(ivl_mult=0.1)), (2,3,4))
test_eq(st._passing_ivls(_ctx._replace(fuzz=0.99, ivl_mult=0.1)), (2,4,6))
test_eq(st._passing_ivls(_ctx._replace(fuzz=0.99, ivl_mult=10.0, max_ivl=5)), (5,5,5))
st2 = ReviewSt(2, 1.3, elapsed=2)
test_eq(st2._passing_ivls(_ctx._replace(hard_mult=0.1)), (1,3,4))

And the shape of the whole machine on the default preset: a new card walks the 1m/10m steps, graduates to 1 day, and a lapsed review re-enters relearning at 10 minutes with its ease floored:

In [ ]:
_nf = mk_ctx(cfg)   # fuzz=None: the deterministic path our oracle tests run under
a = NewSt(0).next_answers(_nf)
test_eq(a.again, LearnSt(2, 60))
test_eq(a.hard, LearnSt(2, 330))
test_eq(a.good, LearnSt(1, 600))
test_eq(a.easy, ReviewSt(4, 2.5))
test_eq(LearnSt(1, 600).next_answers(_nf).good, ReviewSt(1, 2.5))    # last step -> graduate
test_eq(LearnSt(1, 600).next_answers(_nf).again, LearnSt(2, 60))     # Again restarts the ladder

r = ReviewSt(10, 2.5, elapsed=10).next_answers(_nf)
test_eq(r.again, RelearnSt(LearnSt(1, 600), ReviewSt(1, 2.3, 1)))
test_eq((r.hard, r.good, r.easy), (ReviewSt(12, 2.35), ReviewSt(25, 2.5), ReviewSt(32, 2.65)))

rl = RelearnSt(LearnSt(1, 600), ReviewSt(1, 2.3, 1)).next_answers(_nf)
test_eq(rl.good, ReviewSt(1, 2.3, 1))                                 # relearning done: back to review
test_eq(rl.easy, ReviewSt(2, 2.3, 1))
test_eq(rl.again, RelearnSt(LearnSt(1, 600), ReviewSt(1, 2.3, 1)))

Note the Easy interval: `10 * 2.5 * 1.3` looks like 32.5, but the preset's multipliers are protobuf *float32*s (`easy_multiplier` is really 1.2999999523...), so the product lands just under and rounds to 32 -- exactly as Anki's f32 arithmetic has it. Keeping the raw f32 values instead of "tidying" them is part of matching the oracle.

## Reading a card's state

`card_state` recovers the state from a card row's scheduling columns, per `current.rs`. The columns pivot on `type` (0 new, 1 learning, 2 review, 3 relearning) while `queue` picks the due encoding: an intraday learning card (queue 1) stores a unix timestamp, a day-learner (queue 3) or review card a day number. A review card's `elapsed` days come from clamping `due` to today, so an overdue card gets credit for the wait; a learning card's elapsed seconds are reconstructed by re-deriving the fuzzed delay that was added when it was scheduled — recomputable because the fuzz is seeded by `(card, reps)`. Anki reconstructs with its own generator there, so across implementations that reconstruction can differ by the fuzz amount (25%/5min at most); it only feeds displays and FSRS's same-day accounting, never the stored schedule.

In [ ]:
def _card_data(c):
    "The card's `data` column as a dict"
    try: return json.loads(c.data) if c.data else {}
    except ValueError: return {}

def card_state(c, cfg, today, now=None):
    "The scheduling state of card row `c` under preset `cfg`"
    now = ifnone(now, int(time.time()))
    left = c.left%1000
    d = _card_data(c)
    mem = MemSt(d['s'], d['d']) if 's' in d and 'd' in d else None
    if c.type==0: return NewSt(max(c.due,0))
    if c.type==2:
        due = min(c.due, today)
        return ReviewSt(c.ivl, c.factor/1000, c.lapses, elapsed=max(c.ivl-(due-today), 0), mem=mem)
    steps = list(cfg.learn_steps if c.type==1 else cfg.relearn_steps)
    last = step_current_delay(steps, left)
    if c.queue==1: elapsed = now - (c.due - learn_fuzz(c.id+c.reps-1 if c.reps else None, last))
    elif c.queue==3: elapsed = (today - c.due + max(last//DAY, 1))*DAY
    else: elapsed = 0
    learn = LearnSt(left, last, elapsed, mem=mem)
    if c.type==1: return learn
    return RelearnSt(learn, ReviewSt(c.ivl, c.factor/1000, c.lapses, elapsed=c.ivl, mem=mem))

In [ ]:
_now = int(time.time())
crd = lambda **kw: Card(**{**dict(id=1, nid=1, did=1, ord=0, mod=0, usn=-1, type=0, queue=0, due=5, ivl=0), **kw})
test_eq(card_state(crd(), cfg, 10), NewSt(5))
test_eq(card_state(crd(type=2, queue=2, due=8, ivl=7, factor=2500), cfg, 10), ReviewSt(7, 2.5, elapsed=9))
test_eq(card_state(crd(type=2, queue=2, due=12, ivl=7, factor=2500), cfg, 10), ReviewSt(7, 2.5, elapsed=7))   # due is clamped to today: not-yet-due reads as on time
st = card_state(crd(type=1, queue=1, due=_now+300, ivl=0, left=1, reps=2), cfg, 10, now=_now)
test_eq((st.remaining, st.secs), (1, 600))
st = card_state(crd(type=3, queue=3, due=11, ivl=3, factor=2300, left=1, lapses=2), cfg, 10)
test_eq(st.review, ReviewSt(3, 2.3, 2, elapsed=3))
test_eq(st.learn.remaining, 1)

## FSRS

FSRS is a single global switch (the `fsrs` config key), so we always do what the user's other clients would. When it's on, the state machine's `ctx.fsrs` slot carries the four `(memory, interval)` outcomes from `fastanki.fsrs`, computed from the deck preset's parameters. Collections hold up to three parameter generations, and like Anki we prefer version 6, falling back to 5 then 4.5 (`preset_params`); desired retention can be overridden per deck; and whether FSRS may schedule *same-day* steps depends on the short-term parameters being non-zero (`allow_short_term`) plus a config flag for using them alongside explicit steps.

A card that predates FSRS (or was answered by a pre-FSRS client) has no stored memory state, so like Anki we rebuild one by replaying its review log — `_fsrs_reviews` ports `reviews_for_fsrs`'s filtering (drop cramming and manual entries, restart at the last learning sequence or reset, day-granular elapsed times against the next rollover) and `_memory_from_revlog` follows `fsrs_item_for_memory_state`: a complete history replays from scratch, a truncated one starts from an SM-2 approximation of the first surviving entry.

In [ ]:
RLog = namedtuple('RLog', 'id ease ivl lastIvl factor type')

@patch
def fsrs_on(self:Collection):
    "Is FSRS enabled for this collection?"
    return bool(self.conf('fsrs', False))

def preset_params(cfg):
    "The preset's raw FSRS parameters: version 6, else 5, else 4.5, else [] meaning the defaults"
    return list(cfg.fsrs_params_6 or cfg.fsrs_params_5 or cfg.fsrs_params_4)

def allow_short_term(raw):
    "May FSRS schedule same-day steps? Requires non-zero short-term params (default params qualify)"
    if not raw: return True
    return raw[17]>0 and raw[18]>0 if len(raw)>=19 else False

def ignore_revlogs_before(cfg):
    "The preset's ignore-revlogs-before date as epoch ms (0 if unset)"
    s = cfg.ignore_revlogs_before_date
    return int(datetime.strptime(s, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp()*1000) if s else 0

@patch
def _deck_dr(self:Collection, did, cfg):
    "Effective desired retention: the deck's own override, else the preset's"
    k = decks_pb2.Deck.KindContainer()
    k.ParseFromString(self.q1('select kind from decks where id=?', did))
    return k.normal.desired_retention if k.normal.HasField('desired_retention') else cfg.desired_retention

In [ ]:
def _rl_days(e, next_day_at): return max(next_day_at - e.id//1000, 0)//86400

def _fsrs_reviews(entries, next_day_at, ignore_before=0):
    "Filter a card's revlog rows for FSRS and compute (rating, delta_t) pairs; None if nothing usable"
    first_learn = first_grade = None
    for i in reversed(range(len(entries))):
        e = entries[i]
        if e.type==3 and e.factor==0: continue                                   # cramming
        if e.ease>0 and e.id>ignore_before and (e.ivl>=1 or e.ivl<=-86400): first_grade = i
        if e.ease>0 and e.type==0: first_learn = i
        elif e.type==4 and e.factor==0:                                          # reset
            if first_learn is None and first_grade is None: return None
            break
        elif first_learn is not None: break
    complete = first_learn is not None
    if complete and entries[first_learn].id < ignore_before and first_learn < len(entries)-1:
        complete,first_learn = False,None
    start = first_learn if first_learn is not None else first_grade
    if start is None: return None
    kept = [e for e in entries[start:] if e.ease>0 and not (e.type==3 and e.factor==0)]
    if not kept: return None
    ds = [0] + [_rl_days(a, next_day_at)-_rl_days(b, next_day_at) for a,b in zip(kept, kept[1:])]
    return [(e.ease, dt) for e,dt in zip(kept, ds)], complete, kept

def _memory_from_revlog(w, entries, next_day_at, historical_retention=0.9, ignore_before=0):
    "Replay a card's revlog into a `MemSt`, starting a truncated history from an SM-2 approximation"
    out = _fsrs_reviews(entries, next_day_at, ignore_before)
    if out is None: return None
    revs,complete,kept = out
    mem = None
    if not complete:
        first = kept[0]
        ease = (first.factor or 2500)/1000
        mem = memory_state_from_sm2(w, ease, max(first.ivl, 1), historical_retention)
        if ease <= 1.1: mem = mem._replace(difficulty=f32((ease-0.1)*9 + 1))   # entry was written by FSRS itself
        revs = revs[1:]
    for rating,dt in revs: mem = step(w, dt, rating, mem)
    return mem

In [ ]:
@patch
def _fsrs_ctx(self:Collection, c, cfg, next_day):
    "(four FSRS next-states, desired retention, decay) for card `c`, rebuilding memory from the revlog when absent"
    raw = preset_params(cfg)
    w = fsrs_params(raw)
    d = _card_data(c)
    mem = MemSt(d['s'], d['d']) if 's' in d and 'd' in d else None
    if mem is None and c.type!=0:
        rows = [RLog(*r) for r in self.q('select id, ease, ivl, lastIvl, factor, type from revlog where cid=? order by id', c.id)]
        mem = _memory_from_revlog(w, rows, next_day, cfg.historical_retention, ignore_revlogs_before(cfg))
    lrt = d.get('lrt') or self.q1('select max(id)/1000 from revlog where cid=? and ease between 1 and 4 and (type!=3 or factor!=0)', c.id)
    days = max(next_day-lrt, 0)//86400 if lrt else 0
    dr = self._deck_dr(c.did, cfg)
    return next_states(w, mem, dr, days), dr, param_decay(raw)

The pure parts, on a synthetic history: a complete revlog replays to the same state as stepping by hand; a truncated one (no learning entries survive) starts from the SM-2 approximation; manual entries and cramming never count.

In [ ]:
wd = fsrs_params([])
_ms = lambda day: (1700000000+day*86400+3600)*1000  # an hour past the rollover, so same-day reviews share a day count
_nda = 1700000000+900*86400
rl = [RLog(_ms(0), 3, -600, 0, 0, 0), RLog(_ms(0)+600_000, 3, 1, -600, 2500, 0), RLog(_ms(1), 3, 3, 1, 2500, 1)]
byhand = step(wd, 1, 3, step(wd, 0, 3, step(wd, 0, 3, None)))
test_eq(_memory_from_revlog(wd, rl, _nda), byhand)
revs,complete,kept = _fsrs_reviews(rl, _nda)
test_eq((complete, [r for r,_ in revs], [dt for _,dt in revs]), (True, [3,3,3], [0,0,1]))

trunc = [RLog(_ms(0), 3, 10, 5, 2300, 1), RLog(_ms(10), 4, 21, 10, 2450, 1)]
m = _memory_from_revlog(wd, trunc, _nda)
test_eq(m, step(wd, 10, 4, memory_state_from_sm2(wd, 2.3, 10)))

test_is(_memory_from_revlog(wd, [RLog(_ms(0), 0, 5, 5, 0, 4)], _nda), None)  # reset only: nothing usable
test_eq(_fsrs_reviews(rl+[RLog(_ms(2), 0, -1200, 3, 0, 3)], _nda)[0], revs)  # cramming ignored
test_eq(allow_short_term([]), True)
test_eq(allow_short_term([0.1]*17), False)
test_eq(allow_short_term(list(DEFAULT_PARAMS)), True)

## Answering

An answer leaves five marks, each following the sync bookkeeping rules from `fastanki.collection`:
- the card row is rewritten (with `usn=-1` and, for a card leaving the new queue, its original position tucked into the `data` JSON as `pos`, plus the answer time as `lrt`)
- a `revlog` row records the transition (this is the row other clients rebuild from)
- the deck and its parents bump their daily counters, which is how limits stay honest across devices mid-day
- a lapse at the leech threshold tags the note `leech` and optionally suspends the card
- sibling cards get buried per the preset.

Filtered decks stay unsupported, as everywhere in fastanki.

`fuzz=False` turns off both interval fuzz and the learning-delay fuzz, which is what `ANKI_TEST_MODE` does to the oracle — tests run both sides that way and compare exactly.

In [ ]:
@patch
def _card_answers(self:Collection, cid, fuzz, now):
    "Load card `cid` fresh and compute its state, the four answer outcomes, and FSRS extras"
    c = Card(*self.q('select * from cards where id=?', cid)[0])
    assert not c.odid, "cards in filtered decks are not supported"
    now = ifnone(now, int(time.time()))
    today,next_day = self.timing()
    cfg = self.deck_conf(c.did)
    fs = dr = decay = None
    if self.fsrs_on():
        fs,dr,decay = self._fsrs_ctx(c, cfg, next_day)
        raw = preset_params(cfg)
        ctx = mk_ctx(cfg, fuzz=fuzz_factor(c.id, c.reps) if fuzz else None, fsrs=fs, allow_short=allow_short_term(raw),
            short_steps=bool(self.conf('fsrsShortTermWithStepsEnabled', False)))
    else: ctx = mk_ctx(cfg, fuzz=fuzz_factor(c.id, c.reps) if fuzz else None)
    cur = card_state(c, cfg, today, now)
    return c, cfg, cur, cur.next_answers(ctx), today, max(next_day-now, 0), now, dr, decay

@patch
def answer_buttons(self:Collection, card, fuzz=True, now=None):
    "For each ease 1-4, `(next_state, delay_secs)` -- what a client shows on its answer buttons"
    _,_,_,ans,_,sur,_,_,_ = self._card_answers(card.id if isinstance(card,Card) else card, fuzz, now)
    return [(st, ivl_secs(ivl_days(st.ivl_kind(), sur))) for st in ans[1:]]

In [ ]:
@patch
def _leech_note(self:Collection, nid, now):
    tags = self.q1('select tags from notes where id=?', nid).split()
    if 'leech' not in tags:
        self.con.execute('update notes set tags=?, mod=?, usn=-1 where id=?', (f" {' '.join(tags+['leech'])} ", now, nid))

@patch
def _bump_deck_stats(self:Collection, did, today, new_delta, rev_delta, ms_delta):
    "Add an answer to the daily counters of deck `did` and its parents, resetting them on a new day"
    parts = self.q1('select name from decks where id=?', did).split('\x1f')
    names = ['\x1f'.join(parts[:i]) for i in range(1, len(parts)+1)]
    for did_,blob in self.q(f"select id, common from decks where name in ({','.join('?'*len(names))})", *names):
        c = decks_pb2.Deck.Common()
        c.ParseFromString(blob)
        if c.last_day_studied != today:
            c.new_studied,c.learning_studied,c.review_studied,c.milliseconds_studied = 0,0,0,0
            c.last_day_studied = today
        c.new_studied += new_delta
        c.review_studied += rev_delta
        c.milliseconds_studied += ms_delta
        self.con.execute('update decks set common=?, mtime_secs=?, usn=-1 where id=?', (c.SerializeToString(), int(time.time()), did_))

_GATHER_ORD = {1:0, 4:0, 3:1, 2:2, 0:3}  # queue -> gather order: intraday learn, interday learn, review, new

@patch
def _bury_siblings(self:Collection, c, cfg, now):
    "Bury (queue -2) siblings per the preset, only in queues gathered after the answered card's"
    g = _GATHER_ORD.get(c.queue, 99)
    qs = [q for q,want in [(0,cfg.bury_new), (2,cfg.bury_reviews and g<=2), (3,cfg.bury_interday_learning and g<=1)] if want]
    if qs: self.con.execute(f"update cards set queue=-2, mod=?, usn=-1 where nid=? and id!=? and queue in ({','.join('?'*len(qs))})",
        (now, c.nid, c.id, *qs))

In [ ]:
def _shifted_d(mem):
    "FSRS difficulty normalized to the 0.1-1.1 range revlog factors use, x1000"
    return _round(((mem.difficulty-1)/9 + 0.1)*1000)

@patch
def answer_card(self:Collection, card, ease, taken_ms=0, fuzz=True, now=None):
    "Answer `card` (a `Card` or id) with `ease` (1=Again 2=Hard 3=Good 4=Easy), returning the updated `Card`"
    cid = card.id if isinstance(card,Card) else card
    assert ease in (1,2,3,4), f"ease must be 1-4, got {ease}"
    with self._tx():
        c,cfg,cur,ans,today,sur,now,dr,decay = self._card_answers(cid, fuzz, now)
        nxt = ans[ease]
        taken = min(taken_ms, cfg.cap_answer_time_to_secs*1000)
        mem = nxt.learn.mem if isinstance(nxt, RelearnSt) else nxt.mem
        d = _card_data(c)
        if isinstance(cur, NewSt): d['pos'] = cur.position
        d['lrt'] = now
        for k in ('s','d','dr'): d.pop(k, None)
        if mem is not None: d['s'],d['d'] = round(mem.stability, 4), round(mem.difficulty, 3)
        if dr is not None: d['dr'] = round(dr, 2)
        if decay is not None: d['decay'] = round(decay, 3)
        cols = dict(mod=now, usn=-1, reps=c.reps+1, data=json.dumps(d, separators=(',',':')))
        rl_fact = 0
        if isinstance(nxt, ReviewSt):
            cols.update(type=2, queue=2, ivl=nxt.ivl, due=today+nxt.ivl, factor=_round(nxt.ease*1000), lapses=nxt.lapses, left=0)
            rl_fact = _shifted_d(mem) if mem is not None else cols['factor']
        else:
            learn = nxt if isinstance(nxt, LearnSt) else nxt.learn
            if isinstance(nxt, LearnSt):
                cols.update(type=1, left=learn.remaining)
                rl_fact = _shifted_d(mem) if mem is not None else 0
            else:
                cols.update(type=3, left=learn.remaining, ivl=nxt.review.ivl, lapses=nxt.review.lapses, factor=_round(nxt.review.ease*1000))
                rl_fact = _shifted_d(mem) if mem is not None else cols['factor']
            iv = ivl_days(learn.ivl_kind(), sur)
            if iv<0: cols.update(queue=1, due=now+learn_fuzz(c.id+c.reps if fuzz else None, -iv))
            else: cols.update(queue=3, due=today+iv)
        rev = nxt.review if isinstance(nxt, RelearnSt) else nxt
        leeched = isinstance(rev, ReviewSt) and rev.leeched
        if leeched and cfg.leech_action==0: cols['queue'] = -1   # LEECH_ACTION_SUSPEND
        self.con.execute(f"update cards set {', '.join(f'{k}=?' for k in cols)} where id=?", (*cols.values(), cid))
        self.con.execute('insert into revlog values (?,?,?,?,?,?,?,?,?)', (ts_id(self.con,'revlog'), cid, -1,
            ease, ivl_days(nxt.ivl_kind(), sur), ivl_days(cur.ivl_kind(), sur), rl_fact, taken, cur.revlog_kind()))
        self._bump_deck_stats(c.did, today, int(c.queue==0), int(c.queue in (2,3)), taken)
        if leeched: self._leech_note(c.nid, now)
        self._bury_siblings(c, cfg, now)
        self._dirty()
    return Card(*self.q('select * from cards where id=?', cid)[0])

A new card walked through its whole life, checking each mark an answer leaves. Good on a new card enters the second learning step; the revlog interval is `-600` (seconds, by the sign convention), the last interval 0, and the kind 0 (learning):

In [ ]:
n = col.add(Front='sched', Back='uler')
c = col.find_cards(Front='sched')[0]
c1 = col.answer_card(c, 3, taken_ms=1500, fuzz=False)
test_eq((c1.type, c1.queue, c1.left, c1.reps, c1.usn), (1, 1, 1, 1, -1))
assert abs(c1.due - (int(time.time())+600)) <= 3
test_eq(_card_data(c1), dict(pos=c.due, lrt=c1.mod))
rl = col.q('select ease, ivl, lastIvl, factor, time, type, usn from revlog where cid=?', c.id)
test_eq(rl, [(3, -600, 0, 0, 1500, 0, -1)])

Good again graduates it to a 1-day review card; the deck's daily counter has counted one new card; and failing the review tomorrow would drop it into relearning — shown here via `answer_buttons`, which is what a client renders (Again 10m, Hard ~12d, Good 25d... on a 10-day-old review below):

In [ ]:
c2 = col.answer_card(c1, 3, fuzz=False)
test_eq((c2.type, c2.queue, c2.ivl, c2.factor, c2.left, c2.due), (2, 2, 1, 2500, 0, col.today()+1))
test_eq(col.q('select ivl, lastIvl, type from revlog where cid=? order by id', c.id)[-1], (1, -600, 0))
cmn = decks_pb2.Deck.Common()
cmn.ParseFromString(col.q1('select common from decks where id=1'))
test_eq((cmn.new_studied, cmn.last_day_studied, cmn.milliseconds_studied), (1, col.today(), 1500))
test_eq(col.q1('select usn from decks where id=1'), -1)

In [ ]:
col.con.execute('update cards set due=?, ivl=10 where id=?', (col.today(), c.id))   # a 10-day review, due today
btns = col.answer_buttons(c, fuzz=False)
test_eq([s for s,_ in btns], list(ReviewSt(10, 2.5, elapsed=10).next_answers(mk_ctx(cfg))[1:]))
test_eq(btns[0][1], 600)          # Again: 10 minutes of relearning
test_eq(btns[2][1], 25*86400)     # Good: 25 days
c3 = col.answer_card(c, 1, fuzz=False)
test_eq((c3.type, c3.queue, c3.ivl, c3.factor, c3.lapses, c3.left), (3, 1, 1, 2300, 1, 1))
test_eq(col.q('select ivl, lastIvl, factor, type from revlog where cid=? order by id', c.id)[-1], (-600, 10, 2300, 1))
c4 = col.answer_card(c3, 3, fuzz=False)   # relearning done: back to review at the post-lapse interval
test_eq((c4.type, c4.queue, c4.ivl, c4.due), (2, 2, 1, col.today()+1))

## Choosing what to study

`next_card` answers "what now?" for a study session: intraday learning cards whose delay has passed come first, then interday learning and reviews due today (both governed by the review limit), then new cards within the new-card limit, and finally (with nothing else to do), learning cards up to `collapseTime` early. Daily limits subtract the deck counters maintained by `answer_card`, so limits hold across devices and clients. Buried cards from previous days are restored first, the way Anki does when building its queues (deliberately without marking them modified, matching `unbury_on_day_rollover`).

This is a deliberately simplified port of Anki's v3 queue builder: limits come from the *named deck's* preset rather than a per-subdeck limit tree, gathering is by due order (matching Anki's default "Deck" gather with sequential positions), reviews always precede new cards, and there is no display-order matrix. Those affect session ordering, not scheduling state, so they can grow later without compatibility concerns.

In [ ]:
@patch
def unbury_if_day_changed(self:Collection):
    "Restore buried cards once the day has rolled over since the last unbury"
    today,last = self.today(), self.conf('lastUnburied', 0)
    if last < today or today+7 < last:
        for cid,typ,due in self.q('select id, type, due from cards where queue in (-2,-3)'):
            q = {0:0, 2:2}.get(typ, 1 if due>1_000_000_000 else 3)
            self.con.execute('update cards set queue=? where id=?', (q, cid))
        self.con.execute("insert or replace into config values ('lastUnburied',-1,?,?)",
            (now_ms()//1000, json.dumps(today).encode()))

@patch
def _day_limits(self:Collection, did, today):
    "(new_left, review_left) for deck `did` today"
    cfg = self.deck_conf(did)
    cmn = decks_pb2.Deck.Common()
    cmn.ParseFromString(self.q1('select common from decks where id=?', did))
    new_done,rev_done = (cmn.new_studied,cmn.review_studied) if cmn.last_day_studied==today else (0,0)
    new_left,rev_left = max(cfg.new_per_day-new_done, 0), max(cfg.reviews_per_day-rev_done, 0)
    if not self.conf('newCardsIgnoreReviewLimit', False): new_left = min(new_left, rev_left)
    return new_left, rev_left

@patch
def next_card(self:Collection, deck=None, fuzz=True):
    "The next card to study in `deck` (default the whole collection, limits from 'Default'), or None"
    self.unbury_if_day_changed()
    today,now = self.today(), int(time.time())
    new_left,rev_left = self._day_limits(self.deck_id(deck) if deck else 1, today)
    cond,ps = self._find_sql(deck)
    def pick(extra, *xps):
        r = self.q(f'select c.* from cards c join notes n on c.nid=n.id where {cond} and {extra} order by c.due, c.id limit 1', *ps, *xps)
        return Card(*r[0]) if r else None
    c = pick('c.queue=1 and c.due<=?', now)
    if not c and rev_left: c = pick('c.queue=3 and c.due<=?', today)
    if not c and rev_left: c = pick('c.queue=2 and c.due<=?', today)
    if not c and new_left: c = pick('c.queue=0')
    if not c: c = pick('c.queue=1 and c.due<=?', now+self.conf('collapseTime',1200))
    return c

The collection currently holds yesterday's graduate (due tomorrow) and the untouched hola/uno cards from earlier notebooks' pattern — here just the one relearning graduate plus the new cloze pair. A fresh session serves the new cards in position order, and burying kicks in when enabled: answering one cloze sibling hides the other until tomorrow, and `unbury_if_day_changed` brings it back when the day counter moves:

In [ ]:
cz = col.add(model='Cloze', Text='{{c1::sib}} {{c2::ling}}')
nxt = col.next_card()
test_eq((nxt.nid, nxt.ord), (cz.id, 0))  # reviews done for today: first new card, by position

col.con.execute('update deck_config set config=? where id=1',
    (col.deck_conf(1).__class__(bury_new=True).SerializeToString(),))  # minimal preset: bury_new on
cfg1 = col.deck_conf(1)
assert cfg1.bury_new
col.answer_card(nxt, 3, fuzz=False)
sib = col.q('select queue from cards where nid=? and ord=1', cz.id)[0][0]
test_eq(sib, -2)  # sibling buried for the rest of today
test_is(col.next_card() is None or col.next_card().nid != cz.id, True)

col.con.execute("insert or replace into config values ('lastUnburied',-1,0,?)", (json.dumps(col.today()-1).encode(),))
col.unbury_if_day_changed()
test_eq(col.q('select queue from cards where nid=? and ord=1', cz.id)[0][0], 0)

## The oracle test

A collection built by fastanki is copied byte-for-byte, one copy driven by us and the other by Anki, so every id matches and rows compare directly. Before each answer we check **prediction parity**: our four `next_answers` against the oracle's `get_scheduling_states` (elapsed seconds excluded, as Anki itself does when comparing states; ease factors compared at Anki's own stored precision). After each answer, **application parity**: the entire scheduling row, the revlog, the `data` JSON and the deck counters must match, with a few seconds' tolerance only where wall-clock timestamps enter (an intraday due date, the `lrt` stamp).

The script walks every SM-2 path: a new card through both learning steps (with a Hard on the first step to hit the averaging quirk), graduation, passing reviews at each button, a lapse into relearning and out again, Easy graduation straight from learning, and — with the leech threshold dropped to 1 — a lapse that tags the note `leech` on both sides.

In [ ]:
def _canon(st):
    "A state as comparable atoms: ease at Anki's stored precision, elapsed dropped"
    if isinstance(st, NewSt): return ('new', st.position)
    if isinstance(st, LearnSt): return ('learn', st.remaining, st.secs)
    if isinstance(st, ReviewSt): return ('review', st.ivl, _round(st.ease*1000), st.lapses, st.leeched)
    return ('relearn', _canon(st.learn), _canon(st.review))

def _st_of(p):
    "Our state for one of the oracle's `SchedulingState` protos"
    n = p.normal
    k = n.WhichOneof('kind')
    if k=='new': return NewSt(n.new.position)
    if k=='learning': return LearnSt(n.learning.remaining_steps, n.learning.scheduled_secs)
    if k=='review': return ReviewSt(n.review.scheduled_days, n.review.ease_factor, n.review.lapses, leeched=n.review.leeched)
    r,l = n.relearning.review, n.relearning.learning
    return RelearnSt(LearnSt(l.remaining_steps, l.scheduled_secs), ReviewSt(r.scheduled_days, r.ease_factor, r.lapses, leeched=r.leeched))

In [ ]:
_RATING = {1:CardAnswer.AGAIN, 2:CardAnswer.HARD, 3:CardAnswer.GOOD, 4:CardAnswer.EASY}

def _oracle_answer(oc, cid, ease):
    st = oc._backend.get_scheduling_states(cid)
    new_state = [None, st.again, st.hard, st.good, st.easy][ease]
    oc.sched.answer_card(CardAnswer(card_id=cid, current_state=st.current, new_state=new_state,
        rating=_RATING[ease], answered_at_millis=int(time.time()*1000), milliseconds_taken=1500))

def _cmp_card(mycol, oc, cid):
    cols = 'type, queue, ivl, factor, reps, lapses, left, due, data'
    a = mycol.q(f'select {cols} from cards where id=?', cid)[0]
    b = oc.db.execute(f'select {cols} from cards where id=?', cid)[0]
    test_eq(a[:7], tuple(b[:7]))
    if a[1]==1: assert abs(a[7]-b[7])<=5, f'learning due: {a[7]} vs {b[7]}'
    else: test_eq(a[7], b[7])
    da,db_ = json.loads(a[8] or '{}'), json.loads(b[8] or '{}')
    assert abs(da.pop('lrt',0)-db_.pop('lrt',0))<=5
    for k,tol in [('s',0.02),('d',0.02),('dr',1e-6),('decay',1e-6)]:   # float32 vs float64 arithmetic: compare close
        assert abs(da.pop(k,0)-db_.pop(k,0))<=tol, f'{k}: {da} vs {db_}'
    test_eq(da, db_)
    ra = mycol.q('select ease, ivl, lastIvl, factor, type, usn from revlog where cid=? order by id', cid)
    rb = oc.db.execute('select ease, ivl, lastIvl, factor, type, usn from revlog where cid=? order by id', cid)
    test_eq(ra, [tuple(r) for r in rb])

Build the collection, twin it, and walk the script. Every step asserts prediction parity across all four buttons before answering, then answers on both sides and asserts application parity:

In [ ]:
otd = Path(tempfile.mkdtemp())
mycol = Collection.open(otd/'mine'/'collection.anki2')
n1 = mycol.add(Front='oracle', Back='test')
n2 = mycol.add(model='Cloze', Text='{{c1::twin}} {{c2::files}}')
cid1 = mycol.find_card_ids(where='n.id=?', args=[n1.id])[0]
cid2,cid3 = mycol.find_card_ids(where='n.id=?', args=[n2.id])
mycol._clear_wal()
shutil.copy(mycol.path, otd/'theirs.anki2')
oc = AnkiCollection(str(otd/'theirs.anki2'))

def parity_step(mycol, oc, cid, ease):
    "Assert prediction parity on all four buttons, answer on both sides, assert application parity"
    ost = oc._backend.get_scheduling_states(cid)
    _,_,cur,ans,_,_,_,_,_ = mycol._card_answers(cid, False, None)
    for name,op in [('current',ost.current),('again',ost.again),('hard',ost.hard),('good',ost.good),('easy',ost.easy)]:
        test_eq(_canon(getattr(ans, name)), _canon(_st_of(op)))
    mycol.answer_card(cid, ease, taken_ms=1500, fuzz=False)
    _oracle_answer(oc, cid, ease)
    _cmp_card(mycol, oc, cid)

for ease in (3, 2, 3, 3, 1, 3, 4): parity_step(mycol, oc, cid1, ease)   # steps, graduate, review Good, lapse, relearn out, Easy
for ease in (1, 4, 3): parity_step(mycol, oc, cid2, ease)               # Again on new, Easy straight out of learning, review
for ease in (3, 3): parity_step(mycol, oc, cid3, ease)                  # the sibling walks the plain path
print('prediction and application parity across', 7+3+2, 'answers')

prediction and application parity across 12 answers


Deck counters must agree too — and the leech path: drop the threshold to 1 on both sides, lapse a review card, and both implementations must tag the note and record the same lapse:

In [ ]:
cmn_a = decks_pb2.Deck.Common()
cmn_a.ParseFromString(mycol.q1('select common from decks where id=1'))
cmn_b = decks_pb2.Deck.Common()
cmn_b.ParseFromString(bytes(oc.db.execute('select common from decks where id=1')[0][0]))
for f in ('new_studied','review_studied','last_day_studied'): test_eq(getattr(cmn_a,f), getattr(cmn_b,f))

lcfg = mycol.deck_conf(1)
lcfg.leech_threshold = 1
mycol.con.execute('update deck_config set config=? where id=1', (lcfg.SerializeToString(),))
oc.close()
ocon = connect(otd/'theirs.anki2')
ocon.execute('update deck_config set config=? where id=1', (lcfg.SerializeToString(),))
ocon.close()
oc = AnkiCollection(str(otd/'theirs.anki2'))
parity_step(mycol, oc, cid1, 1)
test_eq(mycol.get_note(n1.id).tags, ['leech'])
test_eq(oc.get_note(n1.id).tags, ['leech'])
oc.close()
mycol.close()

And the same but with **FSRS enabled** — including the reconstruction path. The twins first answer one card twice under SM-2, then the `fsrs` config flag is switched on both sides, so that card's memory state must be rebuilt from its revlog while fresh cards walk the learning steps with model-driven graduation. Stored stability/difficulty/retention/decay in the `data` column must match the oracle's at every step:

In [ ]:
ftd = Path(tempfile.mkdtemp())
fcol = Collection.open(ftd/'mine'/'collection.anki2')
fn1 = fcol.add(Front='fsrs', Back='oracle')
fn2 = fcol.add(model='Cloze', Text='{{c1::mem}} {{c2::state}}')
fc1 = fcol.find_card_ids(where='n.id=?', args=[fn1.id])[0]
fc2,fc3 = fcol.find_card_ids(where='n.id=?', args=[fn2.id])
fcol._clear_wal()
shutil.copy(fcol.path, ftd/'theirs.anki2')
foc = AnkiCollection(str(ftd/'theirs.anki2'))
for ease in (3, 3): parity_step(fcol, foc, fc3, ease)   # SM-2 history first: revlog for the reconstruction path

foc.close()
fcon = connect(ftd/'theirs.anki2')
for con in (fcol.con, fcon): con.execute("insert or replace into config values ('fsrs',-1,0,?)", (json.dumps(True).encode(),))
fcon.close()
foc = AnkiCollection(str(ftd/'theirs.anki2'))

for ease in (3, 3, 1, 3): parity_step(fcol, foc, fc1, ease)   # steps and model graduation, lapse, relearn out
for ease in (4, 2): parity_step(fcol, foc, fc2, ease)         # Easy from new, then Hard on the young review
parity_step(fcol, foc, fc3, 3)                                # memory state rebuilt from the SM-2 revlog
sd = _card_data(Card(*fcol.q('select * from cards where id=?', fc3)[0]))
assert 's' in sd and 'd' in sd and sd['dr']==0.9
foc.close()
fcol.close()
print('FSRS parity, including revlog reconstruction')

FSRS parity, including revlog reconstruction
